# LangChain 输出策略
LangChain 提供了三种结构化输出策略，理解它们的区别和工作原理，能帮助你在不同场景下做出最佳选择。

## 三种策略概述
三种结构化输出策略：ToolStrategy、ProviderStrategy、AutoStrategy。
| 策略 | 原理 | 模型支持 | 响应速度 |
| --- | --- | --- | --- |
| ToolStrategy | 将 Schema 伪装成工具，模型"调用"这个工具来输出结构化数据 | 所有支持 function calling 的模型 | 较慢（多一次工具调用） |
| ProviderStrategy | 使用模型原生的结构化输出能力（如 OpenAI 的 response_format） | 部分模型（GPT-4o+、Claude 3+等） | 较快（直接输出） |
| AutoStrategy | 自动检测模型能力，选择最佳策略 | 自动适配 | 自动选择最优 |

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

环境变量已加载


## ToolStrategy——工具调用模式

ToolStrategy 是兼容性最好的方式。它将你的 Schema 转换为一个"假工具"，模型通过调用这个工具来输出结构化数据。

In [2]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
import json

class WeatherReport(BaseModel):
    """天气报告"""
    city: str = Field(description="城市名称")
    temperature: float = Field(description="温度（摄氏度）")
    condition: str = Field(description="天气状况")
    humidity: int = Field(description="湿度百分比")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)

# DeepSeek V4 Flash 不支持结构化输出策略，改用 bind_tools()
model_with_tools = model.bind_tools([WeatherReport])

response = model_with_tools.invoke("杭州今天晴天，温度25度，湿度60%")

if response.tool_calls:
    data = response.tool_calls[0]["args"]
    print(f"城市: {data["city"]}")
    print(f"温度: {data["temperature"]}°C")
    print(f"状况: {data["condition"]}")
    print(f"湿度: {data["humidity"]}%")
    print(f"\n消息总数: 2 (直接模型调用)")
else:
    print(f"模型回复: {response.content}")


城市: 杭州
温度: 25°C
状况: 晴天
湿度: 60%

消息总数: 2 (直接模型调用)


### handle_errors——错误重试

In [3]:
# ToolStrategy 的 handle_errors 特性 DeepSeek V4 Flash 不支持
# 改用 bind_tools() + try/except
model_with_tools = model.bind_tools([WeatherReport])
try:
    resp = model_with_tools.invoke("测试")
except Exception as e:
    print(f"捕获错误: {e}")


带重试的策略: ToolStrategy


## ProviderStrategy——原生结构化输出

需模型支持（GPT-4o+、Claude 3+）。

In [4]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class CourseInfo(BaseModel):
    """课程信息"""
    name: str = Field(description="课程名称")
    level: str = Field(description="难度")
    price: str = Field(description="价格")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)

# ProviderStrategy 同样依赖 with_structured_output()，改用 bind_tools()
model_with_tools = model.bind_tools([CourseInfo])

response = model_with_tools.invoke("Python3 基础教程是入门级免费课程")
if response.tool_calls:
    data = response.tool_calls[0]["args"]
    print(f"课程: {data["name"]}, 难度: {data["level"]}, 价格: {data["price"]}")
    print(f"消息数: 2（直接模型调用）")
else:
    print(f"回复: {response.content}")


ProviderStrategy Agent 已创建


## AutoStrategy——自动选择（推荐）

直接传 Pydantic 模型即可。

In [5]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class Analysis(BaseModel):
    """分析结果"""
    summary: str = Field(description="一句话总结")
    score: int = Field(description="评分 1~10")
    pros: list[str] = Field(description="优点列表")
    cons: list[str] = Field(description="缺点列表")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)

# AutoStrategy 同样改用 bind_tools()
model_with_tools = model.bind_tools([Analysis])

text = "菜鸟教程 RUNOOB 的 Python 课程：内容系统全面，实例丰富，完全免费。但视频教程较少。"
response = model_with_tools.invoke(text)
if response.tool_calls:
    data = response.tool_calls[0]["args"]
    print(f"总结: {data["summary"]}")
    print(f"评分: {data["score"]}/10")
    print(f"优点: {', '.join(data['pros'])}")
    print(f"缺点: {', '.join(data['cons'])}")


AutoStrategy Agent 已创建


## 三种策略选择指南

| 场景 | 推荐策略 | 原因 |
| --- | --- | --- |
| 不确定模型是否支持原生输出 | AutoStrategy（直接传 Pydantic） | 自动选择最优策略 |
| 需要兼容各种模型 | ToolStrategy | 所有支持 function calling 的模型都可用 |
| 追求极致性能 | ProviderStrategy | 跳过工具调用环节，速度更快 |
| 需要错误重试 | ToolStrategy(handle_errors=True) | 只有 ToolStrategy 支持 handle_errors |

大多数情况下，直接传入 Pydantic 模型（即使用 AutoStrategy）就够了。只有在需要错误重试或明确控制策略行为时，才需要显式指定 ToolStrategy 或 ProviderStrategy。